# Lab 5A: CloudWatch Operational Monitoring

In this lab, you'll **monitor the operational health** of your Amazon SageMaker training jobs and inference endpoints using Amazon CloudWatch metrics. You'll explore resource utilization, generate inference traffic, analyze endpoint performance, create a dashboard and alarms, and reason about cost optimization.

## Prerequisites

- Completed **Lab 3A** (Traditional ML experimentation) — you should have a `bank-marketing-xgboost-*` training job and, ideally, a live `bank-marketing-*` endpoint.
- If you deleted your endpoint, you can still analyze **historical training-job metrics** (Sections 2), or redeploy the endpoint from the Lab 3A notebook.

## What you'll do

1. Discover your SageMaker resources (training jobs, endpoints)
2. Analyze **training job metrics** (CPU, memory, disk)
3. Generate **test traffic** against your endpoint
4. Analyze **endpoint metrics** (latency, invocations, errors)
5. Create a **CloudWatch dashboard** (`SageMaker-ML-Operations`)
6. Create **CloudWatch alarms** with SNS notifications
7. Analyze metrics for **cost optimization**
8. (Optional) Clean up

> ⚠️ **Two metric namespaces — don't mix them up:**
> - `AWS/SageMaker` → endpoint invocation metrics (`Invocations`, `ModelLatency`, errors). **`ModelLatency` is in MICROSECONDS** (200 ms = 200,000 µs).
> - `/aws/sagemaker/TrainingJobs` and `/aws/sagemaker/Endpoints` → instance-level utilization (CPU/memory/disk). These are *custom namespaces* in the console metric browser. Utilization can exceed 100% (per-core scale: 400% = 4 vCPUs fully used).


## Two Ways to Investigate: This Notebook (SDK) vs. Kiro CLI (Agent)

This lab supports **two deliberately different experiences** — try both:

| | **This notebook — direct SDK** | **Kiro CLI — monitoring-ops agent** |
|---|---|---|
| Interaction | Explicit `boto3` calls — you see every namespace, dimension, unit, threshold | Natural language — the agent picks the API calls and interprets results |
| Best for | **Engineering**: runbooks, alarms-as-code, learning the CloudWatch data model | **Operations**: exploration, triage, "why is my endpoint slow?" |
| Artifact | Reproducible, reviewable code | A conversation (fast, but not a runbook) |

**Recommended flow**: run this notebook top to bottom first (build the muscle memory — namespaces, the microseconds trap, metric math). Then open a terminal and re-run the *investigations* conversationally.

**Kiro CLI comes pre-installed** in this JupyterLab space — you just need to log in once (device flow, since the terminal can't open a browser):

```bash
kiro-cli login --use-device-flow
```

Open the printed URL in your browser and enter the device code. A **free AWS Builder ID** is perfectly sufficient for this lab (you can create one during sign-in); if you have your own Kiro license (Pro / IAM Identity Center), you can use that instead. Then start the agent:

```bash
kiro-cli chat --agent monitoring-ops
```

Try prompts like *"Was my most recent bank-marketing training job over-provisioned?"* or *"My alarm is stuck in INSUFFICIENT_DATA — what's wrong?"* and compare the two experiences. The lab page has a full section-by-prompt mapping and reflection questions.

> Rule of thumb: **explore and triage with the agent, codify with the SDK** — and never codify something you can't explain.


## Section 1: Setup and Resource Discovery

Initialize clients and find the training jobs and endpoints created in earlier labs.

> **Note on permissions**: this notebook runs as your SageMaker user-profile execution role. It can *read* all metrics and create alarms/SNS topics, but some operations (e.g. creating dashboards) may be restricted — where that happens, we fall back to console instructions.


In [ ]:
# Imports and clients (all pre-installed in SageMaker Distribution)
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone

session = boto3.session.Session()
region = session.region_name
cw = session.client('cloudwatch')
sm = session.client('sagemaker')
smr = session.client('sagemaker-runtime')
sns = session.client('sns')

print(f'Region: {region}')
print('Clients ready: cloudwatch, sagemaker, sagemaker-runtime, sns')

In [ ]:
# 🔗 Console deep-link helpers — build clickable AWS console URLs from this notebook
import urllib.parse
from IPython.display import Markdown, display

_CW = f'https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}'
_SM = f'https://{region}.console.aws.amazon.com/sagemaker/home?region={region}'

def _star_escape(s):
    """CloudWatch graph/Insights fragment encoding: percent-encode everything, then % → * (lowercase hex)."""
    q = urllib.parse.quote(str(s), safe='')
    out, i = '', 0
    while i < len(q):
        if q[i] == '%':
            out += '*' + q[i+1:i+3].lower(); i += 3
        else:
            out += q[i]; i += 1
    return out

def _dollar_escape(s):
    """CloudWatch Logs fragment encoding: double percent-encode, then % → $."""
    return urllib.parse.quote(urllib.parse.quote(str(s), safe='')).replace('%', '$')

def url_metrics_graph(title, *metrics, stat='Average', period=60):
    """Pre-built CloudWatch metrics graph. metrics: tuples of (namespace, metric_name, dims_dict)."""
    m_parts = []
    for ns, mn, dims in metrics:
        p = [f"~'{_star_escape(ns)}", f"~'{_star_escape(mn)}"]
        for k, v in dims.items():
            p += [f"~'{_star_escape(k)}", f"~'{_star_escape(v)}"]
        m_parts.append('~(' + ''.join(p) + ')')
    graph = (f"~(metrics~({''.join(m_parts)})~view~'timeSeries~stacked~false"
             f"~region~'{region}~stat~'{_star_escape(stat)}~period~{period}~title~'{_star_escape(title)})")
    return f'{_CW}#metricsV2:graph={graph}'

def url_log_group(group, stream=None, filter_pattern=None):
    """CloudWatch Logs group (or stream) page; optional pre-set search filter."""
    u = f'{_CW}#logsV2:log-groups/log-group/{_dollar_escape(group)}'
    if stream:
        u += f'/log-events/{_dollar_escape(stream)}'
    if filter_pattern:
        u += f'$3FfilterPattern$3D{_dollar_escape(filter_pattern)}'
    return u

def url_logs_insights(query, group, hours=1):
    """Logs Insights with the query editor and log group pre-filled."""
    qd = (f"~(end~0~start~-{hours*3600}~timeType~'RELATIVE~unit~'seconds"
          f"~editorString~'{_star_escape(query)}~source~(~'{_star_escape(group)}))")
    return f'{_CW}#logsV2:logs-insights$3FqueryDetail$3D{qd}'

def url_dashboard(name):
    return f'{_CW}#dashboards/dashboard/{urllib.parse.quote(name, safe="")}'

def url_alarm(name):
    return f'{_CW}#alarmsV2:alarm/{urllib.parse.quote(name, safe="")}'

def url_sagemaker(kind, name):
    """kind: 'jobs' (training jobs) or 'endpoints'."""
    return f'{_SM}#/{kind}/{urllib.parse.quote(name, safe="")}'

def url_cloudtrail(**filters):
    """CloudTrail Event history, optionally pre-filtered (EventSource=..., EventName=...)."""
    q = urllib.parse.urlencode(filters)
    return (f'https://{region}.console.aws.amazon.com/cloudtrailv2/home?region={region}#/events'
            + (f'?{q}' if q else ''))

def url_eventbridge_rule(name):
    return (f'https://{region}.console.aws.amazon.com/events/home?region={region}'
            f'#/eventbus/default/rules/{urllib.parse.quote(name, safe="")}')

def url_sns_topic(arn):
    return f'https://{region}.console.aws.amazon.com/sns/v3/home?region={region}#/topic/{arn}'

def show_links(items):
    """Render a clickable link list. items: list of (label, url)."""
    display(Markdown('**🔗 Open in AWS Console:**\n' + '\n'.join(f'- [{l}]({u})' for l, u in items)))

print('Console deep-link helpers loaded ✅')

In [ ]:
# Discover recent training jobs
resp = sm.list_training_jobs(SortBy='CreationTime', SortOrder='Descending', MaxResults=10)
training_jobs = resp['TrainingJobSummaries']

print('Recent training jobs:')
for tj in training_jobs:
    print(f"  {tj['TrainingJobName']:60s} {tj['TrainingJobStatus']}")

# Pick the most recent bank-marketing job (fall back to the most recent job of any name)
training_job_name = next(
    (tj['TrainingJobName'] for tj in training_jobs if tj['TrainingJobName'].startswith('bank-marketing')),
    training_jobs[0]['TrainingJobName'] if training_jobs else None,
)
print(f'\nSelected training job: {training_job_name}')

In [ ]:
# Discover endpoints
resp = sm.list_endpoints(SortBy='CreationTime', SortOrder='Descending')
endpoints = resp['Endpoints']

print('Endpoints:')
for ep in endpoints:
    print(f"  {ep['EndpointName']:60s} {ep['EndpointStatus']}")

endpoint_name = next(
    (ep['EndpointName'] for ep in endpoints
     if ep['EndpointName'].startswith('bank-marketing') and ep['EndpointStatus'] == 'InService'),
    None,
)
if endpoint_name is None:
    endpoint_name = next((ep['EndpointName'] for ep in endpoints if ep['EndpointStatus'] == 'InService'), None)

if endpoint_name:
    print(f'\nSelected endpoint: {endpoint_name}')
else:
    print('\n⚠️ No InService endpoint found. Sections 3-6 need a live endpoint.')
    print('   Redeploy from lab3-model-build/lab3a_traditional_ml_experimenation.ipynb,')
    print('   or continue with Section 2 (training-job metrics) only.')

In [ ]:
# Look up the training job's instance details — needed for utilization analysis later
tj_desc = sm.describe_training_job(TrainingJobName=training_job_name)
tj_instance_type = tj_desc['ResourceConfig']['InstanceType']
tj_start = tj_desc['TrainingStartTime']
tj_end = tj_desc.get('TrainingEndTime', datetime.now(timezone.utc))
tj_duration_min = (tj_end - tj_start).total_seconds() / 60

print(f'Training job : {training_job_name}')
print(f'Status       : {tj_desc["TrainingJobStatus"]}')
print(f'Instance     : {tj_instance_type} x {tj_desc["ResourceConfig"]["InstanceCount"]}')
print(f'Started      : {tj_start}')
print(f'Duration     : {tj_duration_min:.1f} minutes')

## Section 2: Training Job Metrics

SageMaker publishes instance-level utilization for training jobs to the **`/aws/sagemaker/TrainingJobs`** namespace with the dimension **`Host`** (value: `<training-job-name>/algo-1`).

| Metric | Use case |
|---|---|
| `CPUUtilization` | Identify CPU bottlenecks (per-core scale — can exceed 100%) |
| `MemoryUtilization` | Detect memory constraints / over-provisioning |
| `GPUUtilization` / `GPUMemoryUtilization` | GPU jobs only |
| `DiskUtilization` | Storage capacity |

We query the exact window when the training job ran.


In [ ]:
# Fetch training-job utilization metrics
def get_training_metric(metric_name, job_name, start, end):
    resp = cw.get_metric_statistics(
        Namespace='/aws/sagemaker/TrainingJobs',
        MetricName=metric_name,
        Dimensions=[{'Name': 'Host', 'Value': f'{job_name}/algo-1'}],
        StartTime=start - timedelta(minutes=2),
        EndTime=end + timedelta(minutes=2),
        Period=60,
        Statistics=['Average', 'Maximum'],
    )
    dps = sorted(resp['Datapoints'], key=lambda d: d['Timestamp'])
    return pd.DataFrame(dps)

training_metrics = {}
for m in ['CPUUtilization', 'MemoryUtilization', 'DiskUtilization', 'GPUUtilization']:
    df = get_training_metric(m, training_job_name, tj_start, tj_end)
    if not df.empty:
        training_metrics[m] = df
        print(f'{m:22s} avg={df.Average.mean():7.2f}%  peak={df.Maximum.max():7.2f}%  ({len(df)} datapoints)')
    else:
        print(f'{m:22s} no data (metric not emitted for this job type)')

In [ ]:
# Plot training-job utilization
if training_metrics:
    fig, ax = plt.subplots(figsize=(12, 5))
    for name, df in training_metrics.items():
        ax.plot(df['Timestamp'], df['Average'], marker='o', label=name)
    ax.set_title(f'Training job utilization — {training_job_name}')
    ax.set_ylabel('Utilization (%) — per-core scale for CPU')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No metric data found. Check that the training job name and region are correct.')

In [ ]:
# 🔗 One-click console links for this training job
show_links([
    (f'SageMaker console — training job {training_job_name}', url_sagemaker('jobs', training_job_name)),
    ('CloudWatch Metrics — CPU & Memory utilization graph', url_metrics_graph(
        f'Training utilization — {training_job_name}',
        ('/aws/sagemaker/TrainingJobs', 'CPUUtilization', {'Host': f'{training_job_name}/algo-1'}),
        ('/aws/sagemaker/TrainingJobs', 'MemoryUtilization', {'Host': f'{training_job_name}/algo-1'}),
    )),
])

### Interpret the numbers

Use this rubric to judge whether the training instance was right-sized:

| Signal | Interpretation | Action |
|---|---|---|
| CPU consistently high (>80% × vCPUs) | Compute used efficiently | Keep or scale up if too slow |
| CPU low (<30%) | Over-provisioned | Use a smaller instance |
| Memory near 100% | OOM risk | Larger instance |
| Memory low (<40%) | Over-provisioned | Smaller instance |
| GPU low (<30%) on GPU instance | Bottleneck elsewhere (CPU/I/O) | Consider CPU instance |

**Remember**: `CPUUtilization` is per-core. On an `ml.m5.xlarge` (4 vCPUs), 100% average means only 1 of 4 cores busy → the *effective* utilization is 25%.


In [ ]:
# Automated right-sizing hint based on observed utilization
VCPUS = {'ml.m5.large': 2, 'ml.m5.xlarge': 4, 'ml.m5.2xlarge': 8, 'ml.m5.4xlarge': 16,
         'ml.c5.xlarge': 4, 'ml.c5.2xlarge': 8, 'ml.g5.2xlarge': 8}
PRICE = {'ml.m5.large': 0.115, 'ml.m5.xlarge': 0.23, 'ml.m5.2xlarge': 0.461,
         'ml.m5.4xlarge': 0.922, 'ml.c5.xlarge': 0.204, 'ml.c5.2xlarge': 0.408}
# Prices are us-east-1 on-demand training prices (USD/hour), indicative only.

if 'CPUUtilization' in training_metrics and tj_instance_type in VCPUS:
    n_vcpu = VCPUS[tj_instance_type]
    cpu_avg = training_metrics['CPUUtilization'].Average.mean()
    effective = cpu_avg / n_vcpu          # normalize per-core scale to 0-100%
    mem_avg = training_metrics.get('MemoryUtilization', pd.DataFrame({'Average': [float('nan')]})).Average.mean()

    print(f'Instance             : {tj_instance_type} ({n_vcpu} vCPUs)')
    print(f'Avg CPUUtilization   : {cpu_avg:.1f}% (per-core scale) → effective {effective:.1f}%')
    print(f'Avg MemoryUtilization: {mem_avg:.1f}%')
    print(f'Training duration    : {tj_duration_min:.1f} min')

    if effective < 30 and mem_avg < 40:
        print('\n💡 Recommendation: instance appears OVER-provisioned.')
        print('   A smaller instance (e.g. one size down) would likely cut training cost ~50%')
        print('   with little impact on wall-clock time for this workload.')
    elif effective > 80 or mem_avg > 90:
        print('\n💡 Recommendation: instance is running hot — consider one size up if jobs are slow or failing.')
    else:
        print('\n💡 Utilization is in a reasonable range for this instance size.')
else:
    print('Not enough data for automated analysis — review the plot above manually.')

## Section 3: Generate Test Traffic

Endpoint invocation metrics **only appear after the endpoint receives requests**. Let's send ~20 inference requests. Metrics take **2–3 minutes** (up to 5) to appear in CloudWatch after invocation.

We reuse a sample row from the Lab 3A bank-marketing dataset (20 label-encoded features, CSV, no header).


In [ ]:
import json
import time

# One valid bank-marketing feature row: 20 label-encoded features, no header and
# no target, in the order the handler's FEATURE_NAMES declares:
#   age, job, marital, education, credit_default, housing, loan, contact,
#   month, day_of_week, duration, campaign, pdays, previous, poutcome,
#   emp_var_rate, cons_price_idx, cons_conf_idx, euribor3m, nr_employed
# pdays=999 is the UCI encoding for "never previously contacted", which is why
# previous=0 and poutcome=1 ('nonexistent') accompany it.
base_row = [56, 1, 1, 1, 0, 0, 0, 1, 4, 2, 180, 2, 999, 0, 1,
            -1.8, 92.893, -46.2, 1.299, 5099.1]

if endpoint_name:
    n_requests = 20
    print(f'Sending {n_requests} requests to {endpoint_name} ...')
    for i in range(n_requests):
        # Sweep call duration, the model's most influential feature, so the
        # traffic spans both classes instead of 20 identical predictions.
        row = list(base_row)
        row[10] = 60 + i * 60
        resp = smr.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='text/csv',
            Body=','.join(str(v) for v in row),
        )
        body = json.loads(resp['Body'].read().decode('utf-8'))
        print(f'  request {i+1:2d}/{n_requests} → duration={row[10]:4d}s  '
              f"class={body['predictions'][0]}  "
              f"P(subscribe)={body['probabilities']['yes'][0]:.4f}")
        time.sleep(2)
    print('\n✅ Traffic generated. Wait 2-3 minutes before querying metrics in the next section.')
    traffic_start = datetime.now(timezone.utc) - timedelta(minutes=2)
else:
    print('⚠️ No endpoint available — skip to Section 7 or redeploy from Lab 3A first.')

## Section 4: Endpoint Metrics

Endpoint invocation metrics live in the **`AWS/SageMaker`** namespace with dimensions **`EndpointName`, `VariantName`**:

| Metric | Description |
|---|---|
| `Invocations` | Number of inference requests |
| `ModelLatency` | Model response time — **microseconds** |
| `OverheadLatency` | SageMaker platform overhead — **microseconds** |
| `Invocation4XXErrors` | Client-side errors |
| `Invocation5XXErrors` | Server-side errors |

Instance-level CPU/memory for the endpoint is in the separate custom namespace `/aws/sagemaker/Endpoints`.


In [ ]:
# Query endpoint invocation metrics for the last 30 minutes
def get_endpoint_metric(metric_name, stat, endpoint, minutes=30, namespace='AWS/SageMaker'):
    end = datetime.now(timezone.utc)
    resp = cw.get_metric_statistics(
        Namespace=namespace,
        MetricName=metric_name,
        Dimensions=[
            {'Name': 'EndpointName', 'Value': endpoint},
            {'Name': 'VariantName', 'Value': 'AllTraffic'},
        ],
        StartTime=end - timedelta(minutes=minutes),
        EndTime=end,
        Period=60,
        Statistics=[stat],
    )
    dps = sorted(resp['Datapoints'], key=lambda d: d['Timestamp'])
    return pd.DataFrame(dps)

if endpoint_name:
    inv = get_endpoint_metric('Invocations', 'Sum', endpoint_name)
    lat = get_endpoint_metric('ModelLatency', 'Average', endpoint_name)
    e4x = get_endpoint_metric('Invocation4XXErrors', 'Sum', endpoint_name)
    e5x = get_endpoint_metric('Invocation5XXErrors', 'Sum', endpoint_name)

    total_inv = int(inv.Sum.sum()) if not inv.empty else 0
    print(f'Invocations (30 min) : {total_inv}')
    if not lat.empty:
        print(f'ModelLatency avg     : {lat.Average.mean():,.0f} µs = {lat.Average.mean()/1000:.1f} ms')
        print(f'ModelLatency max     : {lat.Average.max():,.0f} µs = {lat.Average.max()/1000:.1f} ms')
    print(f'4XX errors           : {int(e4x.Sum.sum()) if not e4x.empty else 0}')
    print(f'5XX errors           : {int(e5x.Sum.sum()) if not e5x.empty else 0}')
    if total_inv == 0:
        print('\n⏳ No datapoints yet — metrics can take up to 5 minutes to appear. Re-run this cell.')

In [ ]:
# Plot invocations and latency side by side
if endpoint_name and not inv.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.bar(inv['Timestamp'], inv['Sum'], width=0.0006)
    ax1.set_title(f'Invocations / minute — {endpoint_name}')
    ax1.set_ylabel('Requests')
    ax1.grid(alpha=0.3)
    if not lat.empty:
        ax2.plot(lat['Timestamp'], lat['Average'] / 1000, marker='o', color='darkorange')
        ax2.set_title('ModelLatency (average)')
        ax2.set_ylabel('Latency (ms)')
        ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# 🔗 One-click console links for this endpoint
if endpoint_name:
    _dims = {'EndpointName': endpoint_name, 'VariantName': 'AllTraffic'}
    show_links([
        (f'SageMaker console — endpoint {endpoint_name}', url_sagemaker('endpoints', endpoint_name)),
        ('CloudWatch Metrics — ModelLatency (µs)', url_metrics_graph(
            f'ModelLatency — {endpoint_name}',
            ('AWS/SageMaker', 'ModelLatency', _dims))),
        ('CloudWatch Metrics — Invocations & errors', url_metrics_graph(
            f'Invocations and errors — {endpoint_name}',
            ('AWS/SageMaker', 'Invocations', _dims),
            ('AWS/SageMaker', 'Invocation4XXErrors', _dims),
            ('AWS/SageMaker', 'Invocation5XXErrors', _dims),
            stat='Sum', period=300)),
        ('CloudWatch Metrics — endpoint CPU/Memory (custom namespace)', url_metrics_graph(
            f'Endpoint utilization — {endpoint_name}',
            ('/aws/sagemaker/Endpoints', 'CPUUtilization', _dims),
            ('/aws/sagemaker/Endpoints', 'MemoryUtilization', _dims))),
    ])

### Interpret endpoint performance

- **ModelLatency**: <100 ms is good for real-time apps; >200 ms needs optimization. For a small XGBoost model expect single-digit ms.
- **Error rate**: 4XX = client issues (malformed payload, wrong content type); 5XX = server issues (model errors, timeouts). Target <1%.
- **Invocations**: watch for traffic patterns → input for capacity planning and auto-scaling.

**Troubleshooting — "I see no metrics"** (check in this order):
1. The endpoint was never invoked → run Section 3.
2. Wrong region in the console → metrics are regional.
3. Wrong namespace → invocation metrics are in `AWS/SageMaker` (AWS namespaces), utilization in `/aws/sagemaker/Endpoints` (custom namespaces).


## Section 5: Create the `SageMaker-ML-Operations` Dashboard

A dashboard gives your team one place to watch latency, traffic, errors, and resource utilization.

We first try to create it programmatically. If your notebook role lacks `cloudwatch:PutDashboard`, the cell prints the exact console steps instead — creating it by hand in the console is part of the learning anyway.

> There is also a built-in operational dashboard at **SageMaker Console → Deployments & inference → Endpoints → (your endpoint)**.


In [ ]:
import json as _json

dashboard_name = 'SageMaker-ML-Operations'

if endpoint_name:
    dims = ['EndpointName', endpoint_name, 'VariantName', 'AllTraffic']
    body = {
        'widgets': [
            {'type': 'metric', 'x': 0, 'y': 0, 'width': 12, 'height': 6, 'properties': {
                'title': 'Endpoint Latency (µs)', 'region': region, 'view': 'timeSeries', 'stat': 'Average', 'period': 60,
                'metrics': [['AWS/SageMaker', 'ModelLatency'] + dims,
                            ['AWS/SageMaker', 'OverheadLatency'] + dims]}},
            {'type': 'metric', 'x': 12, 'y': 0, 'width': 6, 'height': 6, 'properties': {
                'title': 'Total Invocations (5min)', 'region': region, 'view': 'singleValue', 'stat': 'Sum', 'period': 300,
                'metrics': [['AWS/SageMaker', 'Invocations'] + dims]}},
            {'type': 'metric', 'x': 0, 'y': 6, 'width': 12, 'height': 6, 'properties': {
                'title': 'Error Rate (%)', 'region': region, 'view': 'timeSeries', 'period': 300,
                # IF, not MAX: CloudWatch's MAX reduces one time series to a
                # scalar, so MAX([m3,1]) is an unsupported operand type.
                'metrics': [[{'expression': 'IF(m3 > 0, (m1+m2)/m3*100, 0)', 'label': 'Error Rate (%)', 'id': 'e1'}],
                            ['AWS/SageMaker', 'Invocation4XXErrors'] + dims + [{'id': 'm1', 'stat': 'Sum', 'visible': False}],
                            ['AWS/SageMaker', 'Invocation5XXErrors'] + dims + [{'id': 'm2', 'stat': 'Sum', 'visible': False}],
                            ['AWS/SageMaker', 'Invocations'] + dims + [{'id': 'm3', 'stat': 'Sum', 'visible': False}]]}},
            {'type': 'metric', 'x': 12, 'y': 6, 'width': 6, 'height': 6, 'properties': {
                'title': 'Endpoint Resource Utilization', 'region': region, 'view': 'timeSeries', 'stacked': True,
                'stat': 'Average', 'period': 60,
                'metrics': [['/aws/sagemaker/Endpoints', 'CPUUtilization'] + dims,
                            ['/aws/sagemaker/Endpoints', 'MemoryUtilization'] + dims]}},
        ]
    }
    try:
        cw.put_dashboard(DashboardName=dashboard_name, DashboardBody=_json.dumps(body))
        print(f'✅ Dashboard created: {dashboard_name}')
        print(f'   https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards/dashboard/{dashboard_name}')
    except Exception as e:
        print(f'⚠️ Could not create dashboard programmatically: {e}')
        print()
        print('Create it in the console instead:')
        print('  1. CloudWatch Console → Dashboards → Create dashboard → name: SageMaker-ML-Operations')
        print('  2. Add a Line widget: AWS/SageMaker → EndpointName,VariantName → ModelLatency')
        print('  3. Add a Number widget: Invocations (Statistic: Sum, Period: 5 minutes)')
        print('  4. Add a Line widget with metric math IF(m3 > 0, (m1+m2)/m3*100, 0) over 4XX, 5XX,')
        print('     Invocations → "Error Rate (%)"')
        print('  5. Add a Stacked-area widget: /aws/sagemaker/Endpoints → CPUUtilization + MemoryUtilization')
        print('  6. Save dashboard')

## Section 6: Create CloudWatch Alarms

We create an SNS topic `SageMaker-Alerts` and two alarms:

| Alarm | Condition | Notes |
|---|---|---|
| `High-Endpoint-Latency-Alert` | `ModelLatency` avg > **200,000 µs** (= 200 ms), 2 of 3 × 5-min periods | Threshold is in **microseconds!** |
| `High-Error-Rate-Alert` | error rate > 5% over 5 min | Metric math; missing data treated as `notBreaching` (lab endpoints have idle periods) |

> ℹ️ **Anomaly-detection alarms** (as sometimes suggested for `Invocations`) need days of traffic history to train — not useful for a lab endpoint that is minutes old. We use static thresholds.


In [ ]:
# Create SNS topic and (optionally) subscribe your email
topic_name = 'SageMaker-Alerts'
topic_arn = sns.create_topic(Name=topic_name)['TopicArn']   # idempotent
print(f'SNS topic: {topic_arn}')

# 👇 OPTIONAL: put your email here to receive alarm notifications
notification_email = ''   # e.g. 'you@example.com'

if notification_email:
    sns.subscribe(TopicArn=topic_arn, Protocol='email', Endpoint=notification_email)
    print(f'Subscription requested for {notification_email}.')
    print('📧 IMPORTANT: check your inbox (and spam) and CONFIRM the subscription,')
    print('   otherwise no notifications will be delivered.')
else:
    print('No email configured — alarms will still change state, just without notifications.')

In [ ]:
# Alarm 1: High endpoint latency (threshold in MICROSECONDS)
if endpoint_name:
    cw.put_metric_alarm(
        AlarmName='High-Endpoint-Latency-Alert',
        AlarmDescription='Alert when endpoint ModelLatency exceeds 200ms (200,000 microseconds)',
        Namespace='AWS/SageMaker',
        MetricName='ModelLatency',
        Dimensions=[
            {'Name': 'EndpointName', 'Value': endpoint_name},
            {'Name': 'VariantName', 'Value': 'AllTraffic'},
        ],
        Statistic='Average',
        Period=300,
        EvaluationPeriods=3,
        DatapointsToAlarm=2,
        Threshold=200000,              # microseconds!
        ComparisonOperator='GreaterThanThreshold',
        TreatMissingData='notBreaching',
        AlarmActions=[topic_arn],
    )
    print('✅ Alarm created: High-Endpoint-Latency-Alert (>200,000 µs = 200 ms)')

In [ ]:
# Alarm 2: High error rate (>5%) using metric math
if endpoint_name:
    dims = [
        {'Name': 'EndpointName', 'Value': endpoint_name},
        {'Name': 'VariantName', 'Value': 'AllTraffic'},
    ]
    cw.put_metric_alarm(
        AlarmName='High-Error-Rate-Alert',
        AlarmDescription='Alert when endpoint error rate exceeds 5%',
        Metrics=[
            # Guard the divide-by-zero with IF, not MAX: CloudWatch's MAX takes a
            # single time series and returns a scalar, so MAX([m3,1]) — mixing a
            # series with a constant — is rejected as an unsupported operand type.
            {'Id': 'e1', 'Expression': 'IF(m3 > 0, (m1+m2)/m3*100, 0)', 'Label': 'Error Rate (%)', 'ReturnData': True},
            {'Id': 'm1', 'ReturnData': False, 'MetricStat': {'Stat': 'Sum', 'Period': 300, 'Metric': {
                'Namespace': 'AWS/SageMaker', 'MetricName': 'Invocation4XXErrors', 'Dimensions': dims}}},
            {'Id': 'm2', 'ReturnData': False, 'MetricStat': {'Stat': 'Sum', 'Period': 300, 'Metric': {
                'Namespace': 'AWS/SageMaker', 'MetricName': 'Invocation5XXErrors', 'Dimensions': dims}}},
            {'Id': 'm3', 'ReturnData': False, 'MetricStat': {'Stat': 'Sum', 'Period': 300, 'Metric': {
                'Namespace': 'AWS/SageMaker', 'MetricName': 'Invocations', 'Dimensions': dims}}},
        ],
        EvaluationPeriods=2,
        DatapointsToAlarm=2,
        Threshold=5,
        ComparisonOperator='GreaterThanThreshold',
        TreatMissingData='notBreaching',   # idle lab endpoints have empty periods
        AlarmActions=[topic_arn],
    )
    print('✅ Alarm created: High-Error-Rate-Alert (>5%)')

In [ ]:
# Check alarm states
resp = cw.describe_alarms(AlarmNames=['High-Endpoint-Latency-Alert', 'High-Error-Rate-Alert'])
for a in resp['MetricAlarms']:
    print(f"{a['AlarmName']:35s} state={a['StateValue']:20s} ({a['StateReason'][:70]})")

print()
print('State meanings:')
print('  OK                → metric within threshold')
print('  INSUFFICIENT_DATA → not enough datapoints yet (normal right after creation; generate traffic and wait)')
print('  ALARM             → threshold breached; SNS notification sent (if subscription confirmed)')

In [ ]:
# 🔗 Jump straight to the dashboard and alarms you just created
show_links([
    ('CloudWatch dashboard — SageMaker-ML-Operations', url_dashboard('SageMaker-ML-Operations')),
    ('Alarm — High-Endpoint-Latency-Alert', url_alarm('High-Endpoint-Latency-Alert')),
    ('Alarm — High-Error-Rate-Alert', url_alarm('High-Error-Rate-Alert')),
])

## Section 7: Cost Optimization Analysis

CloudWatch metrics are your primary input for right-sizing decisions. Two levers:

**1. Right-size the instance** — compare endpoint CPU/Memory utilization against instance size:

| Scenario | Recommendation |
|---|---|
| <30% CPU/Memory sustained | One size down (≈50% saving) |
| >80% CPU/Memory or latency spikes | One size up, or add auto-scaling |

**2. Match capacity to traffic patterns** — from the `Invocations` history:
- Predictable peaks → **auto-scaling** on `InvocationsPerInstance`
- Long idle periods → **serverless inference** for sporadic traffic

**Example ROI calculation** (analysis only — we don't resize anything in this lab):

```
Current : ml.m5.xlarge × 2 (HA)          → $0.269/h × 2 × 8760 h = $4,713/year
Optimized: ml.m5.large × 1 + auto-scaling → $0.134/h × ~1.3 × 8760 h = $1,526/year
Annual saving: ~$3,187 (68%)
```


In [ ]:
# Endpoint utilization snapshot (instance-level metrics, custom namespace)
if endpoint_name:
    cpu = get_endpoint_metric('CPUUtilization', 'Average', endpoint_name, minutes=60,
                              namespace='/aws/sagemaker/Endpoints')
    mem = get_endpoint_metric('MemoryUtilization', 'Average', endpoint_name, minutes=60,
                              namespace='/aws/sagemaker/Endpoints')
    ep_desc = sm.describe_endpoint(EndpointName=endpoint_name)
    ep_cfg = sm.describe_endpoint_config(EndpointConfigName=ep_desc['EndpointConfigName'])
    variant = ep_cfg['ProductionVariants'][0]
    print(f"Endpoint instance: {variant.get('InstanceType', 'serverless')} x {variant.get('InitialInstanceCount', '-')}")
    if not cpu.empty:
        print(f'CPU avg (60 min)  : {cpu.Average.mean():.1f}%  (per-core scale)')
    if not mem.empty:
        print(f'Memory avg (60 min): {mem.Average.mean():.1f}%')
    if cpu.empty and mem.empty:
        print('No utilization datapoints yet — these appear a few minutes after the endpoint serves traffic.')
    else:
        print()
        print('💡 A lab endpoint serving sporadic test traffic will show near-zero utilization —')
        print('   in production, collect 1-2 weeks of data before making right-sizing decisions.')

## Section 8: (Optional) Cleanup

The alarms, dashboard, and SNS topic cost almost nothing, and **Lab 5B reuses the `SageMaker-Alerts` topic** — only clean up if you're done with all of Lab 5.

> ⚠️ Do **NOT** delete your SageMaker endpoint here — Labs 5B and 5C use it.


In [ ]:
# Uncomment to clean up Lab 5A resources (keep them if you continue with Lab 5B/5C!)

# cw.delete_alarms(AlarmNames=['High-Endpoint-Latency-Alert', 'High-Error-Rate-Alert'])
# cw.delete_dashboards(DashboardNames=['SageMaker-ML-Operations'])
# sns.delete_topic(TopicArn=topic_arn)
# print('Lab 5A monitoring resources deleted')

## Key Takeaways

✅ **Training-job metrics** (`/aws/sagemaker/TrainingJobs`, dimension `Host`) reveal whether training instances are right-sized — remember the per-core CPU scale.
✅ **Endpoint metrics** (`AWS/SageMaker`, dimensions `EndpointName`+`VariantName`) track latency, traffic, and errors — **`ModelLatency` is in microseconds**.
✅ Metrics only exist **after invocations**; allow 2–5 minutes of propagation.
✅ A **dashboard** unifies latency, traffic, error rate (metric math), and utilization.
✅ **Alarms → SNS** enable proactive alerting; use `notBreaching` for low-traffic endpoints and static thresholds (not anomaly detection) for young endpoints.
✅ Utilization + invocation patterns drive **cost optimization** (right-sizing, auto-scaling, serverless).

## Next Steps

Continue to **Lab 5B: CloudWatch Logs Monitoring** (`lab5b-cloudwatch-logs-monitoring.ipynb`) to debug and troubleshoot your SageMaker workloads using training and endpoint logs.
